# Importing Libraries 

In [1]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from catboost import CatBoostClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, log_loss, roc_auc_score

# Load datasets

In [2]:
train_df = pd.read_csv('dg_incident_train.csv')
val_df = pd.read_csv('dg_incident_validation.csv')
test_df = pd.read_csv('dg_incident_test.csv')

In [3]:
print(train_df.head())

   record_id            timestamp shc_code origin_destination  dg_class  \
0  DG_000001  2025-11-04 07:53:23      RFL            AMS-JED       4.0   
1  DG_000002  2025-04-13 05:40:50      MAG            MXP-JED       2.1   
2  DG_000003  2025-03-10 17:07:01      CAO            AMS-JED       2.1   
3  DG_000004  2025-06-15 19:27:32      ELI            OSL-JED       8.0   
4  DG_000005  2025-06-04 02:33:27      ELI            AMS-JED       2.1   

  packaging_type  handling_error_count  previous_incident_count  \
0        Plastic                     2                        3   
1          Fiber                     4                        1   
2          Metal                     4                        1   
3          Fiber                     4                        6   
4        Plastic                     5                        2   

   cargo_weight_kg  temperature_celsius  humidity_percentage  \
0         36366.21                35.58                36.94   
1         38195.46

# data cleaning and feature engineering

In [4]:
# --- PHASE 1: DATA CLEANING ---
def clean_data(df):
    # Make a copy to avoid changing the original raw data
    df_clean = df.copy()
    
    # 1. Handle Missing Values
    for col in df_clean.columns:
        if df_clean[col].isnull().sum() > 0:
            if df_clean[col].dtype == 'object' or df_clean[col].dtype.name == 'category':
                # Filling categorical with mode
                df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])
            else:
                # Filling numerical with median
                df_clean[col] = df_clean[col].fillna(df_clean[col].median())
    
    # 2. Format Timestamps & Feature Expansion
    df_clean['timestamp'] = pd.to_datetime(df_clean['timestamp'])
    df_clean['shipment_hour'] = df_clean['timestamp'].dt.hour
    df_clean['shipment_day_of_week'] = df_clean['timestamp'].dt.dayofweek # New: Captures weekend risk
    df_clean['shipment_month'] = df_clean['timestamp'].dt.month
    
    # 3. Type Casting for CatBoost
    # CatBoost requires categorical columns to be explicitly strings or categories
    cat_cols = ['shc_code', 'origin_destination', 'packaging_type', 'weather_condition']
    for col in cat_cols:
        df_clean[col] = df_clean[col].astype(str)
    
    # 4. Remove "Noise" 
    # record_id and shipper_id are dropped. timestamp is dropped AFTER extraction.
    noise_cols = ['record_id', 'shipper_id', 'timestamp']
    df_clean = df_clean.drop(columns=noise_cols)
    
    return df_clean

# Apply cleaning
train_cleaned = clean_data(train_df)
val_cleaned = clean_data(val_df)
test_cleaned = clean_data(test_df)

print("Cleaning Complete for CatBoost.")

Cleaning Complete for CatBoost.


In [5]:
train_cleaned.head()

,shc_code,origin_destination,dg_class,packaging_type,handling_error_count,previous_incident_count,cargo_weight_kg,temperature_celsius,humidity_percentage,weather_condition,safety_staff_count,doc_audit_result,incident_flag,shipment_hour,shipment_day_of_week,shipment_month
0,RFL,AMS-JED,4.0,Plastic,2,3,36366.21,35.58,36.94,Extreme Heat,31,1,0,7,1,11
1,MAG,MXP-JED,2.1,Fiber,4,1,38195.46,41.83,10.03,Fog,29,0,0,5,6,4
2,CAO,AMS-JED,2.1,Metal,4,1,40098.77,15.72,38.36,Storm,7,1,1,17,0,3
3,ELI,OSL-JED,8.0,Fiber,4,6,3589.44,-11.79,15.58,Rain,1,0,0,19,6,6
4,ELI,AMS-JED,2.1,Plastic,5,2,32940.57,10.63,67.59,Clear,47,0,1,2,2,6


In [6]:
# --- PHASE 1: FEATURE ENGINEERING ---

def engineer_features(df):
    df_eng = df.copy()
    
    # 1. Mishandling Threshold
    df_eng['is_critical_mishandling'] = (df_eng['handling_error_count'] > 7).astype(int)
    
    # 2. Climate Shock Risk (OSL-JED + Temperature)
    # Captures the thermal expansion risk for specific route and high temperature
    df_eng['climate_shock_risk'] = (
        (df_eng['origin_destination'] == 'OSL-JED') & 
        (df_eng['temperature_celsius'] > 30)
    ).astype(int)
    
    # 3. Trusted Shipper Status
    # High frequency of past violations marks an untrusted shipper
    df_eng['is_untrusted_shipper'] = (df_eng['previous_incident_count'] > 5).astype(int)
    
    return df_eng

# Apply to your cleaned dataframes
train_final = engineer_features(train_cleaned)
val_final = engineer_features(val_cleaned)
test_final = engineer_features(test_cleaned)

print("Feature Engineering Complete.")
print(f"New Features Added: is_critical_mishandling, climate_shock_risk, is_untrusted_shipper")

Feature Engineering Complete.
New Features Added: is_critical_mishandling, climate_shock_risk, is_untrusted_shipper


In [7]:
train_final.head()

,shc_code,origin_destination,dg_class,packaging_type,handling_error_count,previous_incident_count,cargo_weight_kg,temperature_celsius,humidity_percentage,weather_condition,safety_staff_count,doc_audit_result,incident_flag,shipment_hour,shipment_day_of_week,shipment_month,is_critical_mishandling,climate_shock_risk,is_untrusted_shipper
0,RFL,AMS-JED,4.0,Plastic,2,3,36366.21,35.58,36.94,Extreme Heat,31,1,0,7,1,11,0,0,0
1,MAG,MXP-JED,2.1,Fiber,4,1,38195.46,41.83,10.03,Fog,29,0,0,5,6,4,0,0,0
2,CAO,AMS-JED,2.1,Metal,4,1,40098.77,15.72,38.36,Storm,7,1,1,17,0,3,0,0,0
3,ELI,OSL-JED,8.0,Fiber,4,6,3589.44,-11.79,15.58,Rain,1,0,0,19,6,6,0,0,1
4,ELI,AMS-JED,2.1,Plastic,5,2,32940.57,10.63,67.59,Clear,47,0,1,2,2,6,0,0,0


# Data Encoding

In [8]:
# --- PHASE 1: DATA FINALIZATION ---

def finalize_phase_1(df):
    df_final = df.copy()
    
    # --- 1. PATTERN LOGIC: THE "DANGEROUS COMBINATIONS" ---
    
    # Combination: Physical Abuse + High Hazard (Class 2.1 is Flammable Gas)
    df_final['gas_handling_risk'] = ((df_final['dg_class'] == 2.1) & 
                                     (df_final['handling_error_count'] > 5)).astype(int)
    
    # Combination: Climate Shock
    df_final['thermal_expansion_risk'] = ((df_final['origin_destination'] == 'OSL-JED') & 
                                          (df_final['temperature_celsius'] > 35) & 
                                          (df_final['dg_class'] == 3.0)).astype(int)

    # Combination: Untrusted Shipper + High Hazard SHC
    df_final['shipper_hazard_combo'] = ((df_final['previous_incident_count'] > 5) & 
                                        (df_final['shc_code'] == 'CAO')).astype(int)

    # --- 2. TYPE CASTING FOR CATBOOST ---
    # Instead of Encoding, we ensure all categorical features are 'string' or 'object'
    # CatBoost will perform its own 'Ordered Boosting' encoding internally.
    categorical_cols = ['shc_code', 'origin_destination', 'packaging_type', 'weather_condition']
    
    for col in categorical_cols:
        df_final[col] = df_final[col].fillna('Unknown').astype(str)
        
    return df_final

# Apply to your current dataframes
train_ready = finalize_phase_1(train_final)
val_ready = finalize_phase_1(val_final)
test_ready = finalize_phase_1(test_final)

print("Phase 1 Successfully Completed for CatBoost!")
print(f"Total Features for the Model: {train_ready.shape[1] - 1}")

Phase 1 Successfully Completed for CatBoost!
Total Features for the Model: 21


# Exploratory Data Analysis

In [9]:
# EDA code continues here